# 03 — U-Net Eğitimi: Aşama 0 (keşif & kurulum)

**Amaç:** Organizatörün DL baseline'ını (`TemporalUNet3D` + `SimpleNodeTransformer`) Kaggle'da
eğitebildiğimizi doğrulamak ve **süre/bellek bütçesini ölçmek**. README diyor ki *"not trained
to convergence — expect gains from training longer"* → asıl kaldıraç budur.

Bu Aşama 0'ın çıktısı tüm eğitim planını netleştirir:
- GPU gerçekten açık mı? (kaç GPU, ne)
- Kurulum sorunsuz mu (tracksdata, zarr, repo)?
- **epoch/saat** ve **GPU bellek** (kaç epoch haftalık kotaya sığar?)
- `downsample=1,2,2` (yüksek çözünürlük, instance-ayrımı) belleğe sığar mı?
- Checkpoint formatı inference notebook ile uyumlu mu?

> ### ⚠️ NOTEBOOK AYARLARI
> - **Accelerator = GPU T4 x2**
> - **Internet = ON** (bu notebook submit EDİLMEZ; git clone + pip gerekiyor)
> - **Add Input:** yarışma verisi (`biohub-cell-tracking-during-development`)

## 1 — GPU kontrolü (önce bunu netleştir)

In [ ]:
import torch, subprocess
print('torch', torch.__version__)
print('CUDA var mi:', torch.cuda.is_available())
print('GPU sayisi:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU{i}: {p.name} | {p.total_memory/1e9:.1f} GB')
assert torch.cuda.is_available(), 'GPU YOK — Accelerator = GPU T4 x2 sec!'
print('\n', subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout[:600])

## 2 — Repoyu klonla (royerlab, eğitim kodu)

In [ ]:
import os
from pathlib import Path
REPO = Path('/kaggle/working/tracking_repo')
if not REPO.exists():
    !git clone --depth 1 https://github.com/royerlab/kaggle-cell-tracking-competition.git {REPO}
print('\nrepo dosyalari:')
!ls {REPO}/scripts {REPO}/src/tracking_cellmot/models

## 3 — Bağımlılıklar (torch'a DOKUNMA — CUDA build'i koru)

Sadece eksikleri kur: `tracksdata` (git), `zarr>=3.0.10`, `scipy`, `tqdm`. Torch'u
yeniden kurdurma (CUDA build'i bozulur, çok yavaş). Repoyu `PYTHONPATH` ile kullan
(`pip install -e .` torch>=2.9.1 isteyip torch'u upgrade edebilir — kaçın).

In [ ]:
# tracksdata'yi bagimliliksiz kur (torch/numpy CUDA build'ine dokunma), sonra tracksdata'nin
# GEREKTIRDIGI paketleri ekle: geff, bidict, ilpy, imagecodecs, polars>=1.36, rustworkx.
# (Ilk kosuda 'import geff' patlamisti; --no-deps bunlari atlamisti.)
!pip install -q --no-deps 'tracksdata @ git+https://github.com/royerlab/tracksdata@main'
!pip install -q 'geff>=1.1.3.1.1' 'bidict>=0.23.1' imagecodecs 'polars>=1.36' 'ilpy>=0.5.1' rustworkx 'zarr>=3.0.10' tqdm

import sys, importlib
sys.path[:0] = [str(REPO / 'src'), str(REPO / 'scripts')]
for m in ['zarr', 'geff', 'bidict', 'ilpy', 'imagecodecs', 'polars', 'rustworkx', 'tracksdata', 'scipy', 'tqdm']:
    try:
        mod = importlib.import_module(m)
        v = getattr(mod, '__version__', '?')
        print('  OK', m, v)
    except Exception as e:
        print('  HATA', m, ':', type(e).__name__, e)
import tracking_cellmot
from tracking_cellmot.models.temporal_unet import TemporalUNet3D
from tracking_cellmot.models.simple_node_transformer import SimpleNodeTransformer
print('  OK repo modulleri (TemporalUNet3D, SimpleNodeTransformer)')

## 4 — Veri mount kontrolü

`dataspec.py` Kaggle train mount'unu otomatik bulur; splits dosyası yoksa 90/10 seed-0
split üretir. Yani `--split 0` doğrudan çalışır. Sadece verinin ekli olduğunu doğrula.

In [ ]:
TRAIN = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
assert TRAIN.exists(), 'Yarisma verisi ekli degil! Add Input -> yarisma'
zarrs = sorted(TRAIN.glob('*.zarr')); geffs = sorted(TRAIN.glob('*.geff'))
print(f'{len(zarrs)} .zarr, {len(geffs)} .geff  (eslesme: {len(set(z.stem for z in zarrs) & set(g.stem for g in geffs))})')

## 5 — MİNİK eğitim: 1 epoch, birkaç iter → süre/bellek ölç

`--max-iters` ile birkaç adım koş; amaç skor değil **uçtan uca çalışıyor mu + hız + bellek**.
`--num-workers 2` (Kaggle CPU sınırlı). İki T4'ü `--data-parallel` (varsayılan) kullanır.

In [ ]:
import time, os
env = dict(os.environ, PYTHONPATH=f"{REPO}/src:{REPO}/scripts")
cmd = [
    'python', str(REPO / 'scripts' / 'train_unet_transformer.py'),
    '--split', '0', '--epochs', '1', '--max-iters', '20',
    '--batch-size', '8', '--num-workers', '2',
]
print('KOMUT:', ' '.join(cmd), '\n')
t0 = time.time()
r = subprocess.run(cmd, cwd=str(REPO), env=env, capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print('--- STDERR ---\n', r.stderr[-3000:])
print(f'\n>> 20 iter suresi: {time.time()-t0:.0f} s | returncode={r.returncode}')
print(subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout)

## 6 — `downsample=1,2,2` bellek testi (yüksek çözünürlük hipotezi)

Klasik hattı yenen **instance-ayrımı** (255 zor-FN) için daha az downsample = daha yüksek
çözünürlük. Ama 4× bellek. Birkaç iter koşup OOM olup olmadığına bak.

In [ ]:
cmd2 = [
    'python', str(REPO / 'scripts' / 'train_unet_transformer.py'),
    '--split', '0', '--epochs', '1', '--max-iters', '10',
    '--batch-size', '4', '--num-workers', '2', '--downsample', '1,2,2',
]
t0 = time.time()
r2 = subprocess.run(cmd2, cwd=str(REPO), env=env, capture_output=True, text=True)
print(r2.stdout[-1500:])
if r2.returncode != 0:
    tail = r2.stderr[-1500:]
    print('--- STDERR ---\n', tail)
    print('\n>> OOM mu?', 'out of memory' in tail.lower() or 'CUDA' in tail)
print(f'>> downsample 1,2,2 | 10 iter {time.time()-t0:.0f}s | returncode={r2.returncode}')

## 7 — Checkpoint formatı doğrula (inference ile uyum)

In [ ]:
ckpt = REPO / 'weights' / 'unet_transformer' / 'split_0' / 'edge_predictor_best.pth'
print('checkpoint var mi:', ckpt.exists(), '|', ckpt if ckpt.exists() else '')
if ckpt.exists():
    sd = torch.load(ckpt, map_location='cpu', weights_only=True)
    keys = list(sd.keys())
    print(f'{len(keys)} tensor | boyut {ckpt.stat().st_size/1e6:.1f} MB')
    print('unet anahtarlari:', sum(k.startswith('unet.') for k in keys),
          '| transformer:', sum(not k.startswith('unet.') for k in keys))
    print('ornek:', keys[:3], '...', keys[-2:])
    print('\n>> inference notebook `weights/unet_transformer/split_0/edge_predictor_best.pth` bekliyor — UYUMLU')

## 8 — Aşama 0 sonucu & sonraki adım

Bu koşudan çıkması gerekenler (bana at):
- **GPU:** kaç adet, ne (T4 x2 mi?)
- **20 iter süresi** → epoch başına ~kaç dk (bir epoch'ta iter sayısı = train dataset × pencere)
- **GPU bellek** (default vs downsample 1,2,2)
- **downsample 1,2,2 OOM mu?**
- **checkpoint** oluştu ve uyumlu mu?

Buna göre **Aşama 1** planı:
- Kotaya sığan epoch sayısıyla tam eğitim (checkpoint + resume)
- Hipotez A: default ayarla daha uzun eğit · Hipotez B: `downsample 1,2,2` yüksek çözünürlük
- Her checkpoint'i `scripts/evaluate.py` (resmi metrik) ile birkaç train dataset'te ele → en iyisini inference notebook'a tak → **submit yakmadan** karşılaştır.